In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.base import BaseEstimator, OutlierMixin
from typing import Optional


from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np


class TransactionIsoForestModel:
    def __init__(
        self,
        n_estimators=200,
        contamination="auto",
        max_samples="auto",
        random_state=42,
    ):
        """
        Skeleton for an Isolation Forest anomaly detection pipeline.
        """
        self.scaler = StandardScaler()
        self.model = IsolationForest(
            n_estimators=n_estimators,
            contamination=contamination,
            max_samples=max_samples,
            random_state=random_state,
            n_jobs=-1
        )
        self.fitted = False

    # ----------------------------------------------------------
    # STEP 1: Feature Engineering (placeholder)
    # ----------------------------------------------------------
    def _preprocess_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Insert your feature engineering here.
        df is raw transaction data.
        """

        # Example placeholder features:
        df = df.copy()
        df["TransactionDate"] = pd.to_datetime(df["TransactionDate"])
        df["PreviousTransactionDate"] = pd.to_datetime(df["PreviousTransactionDate"])

        # Time-based features
        df["Trans_Hour"] = df["TransactionDate"].dt.hour
        df["Trans_Day"] = df["TransactionDate"].dt.day
        df["Trans_Month"] = df["TransactionDate"].dt.month
        df["Trans_DOW"] = df["TransactionDate"].dt.weekday
        df["Trans_Year"] = df["TransactionDate"].dt.year

        # Time since previous transaction (proxy for frequency)
        df["Hours_Since_Prev"] = (
            (df["TransactionDate"] - df["PreviousTransactionDate"])
            .dt.total_seconds() / 3600
        )

        # -----------------------------
        # Frequency-based features per account
        # -----------------------------
        df["Account_Transaction_Count"] = df.groupby("AccountID")["TransactionID"].transform("count")

        df["Account_Trans_Freq_Per_Month"] = (
            df.groupby("AccountID")["TransactionDate"]
            .transform(lambda x: len(x) / (x.max() - x.min()).days * 30
                      if (x.max() - x.min()).days > 0 else 0)
        )

        # -----------------------------
        # Monetary features
        # -----------------------------
        df["Log_TransactionAmount"] = np.log1p(df["TransactionAmount"])

        # Cumulative spending per year (proxy)
        df["Cumulative_Spending_Year"] = df.groupby(
            ["AccountID", df["TransactionDate"].dt.year]
        )["TransactionAmount"].cumsum()

        # Average spending for this customer
        df["Avg_TransactionAmount_Account"] = df.groupby("AccountID")["TransactionAmount"].transform("mean")

        # Deviation from customer baseline (fraud psychometric feature)
        df["Amount_Deviation"] = df["TransactionAmount"] - df["Avg_TransactionAmount_Account"]

        # -----------------------------
        # Device / IP / Location risk encodings
        # -----------------------------
        df["DeviceID_Freq"] = df.groupby("DeviceID")["TransactionID"].transform("count")
        df["IP_Freq"] = df.groupby("IP Address")["TransactionID"].transform("count")
        df["Location_Freq"] = df.groupby("Location")["TransactionID"].transform("count")

        # -----------------------------
        # Merchant (Vendor) features
        # -----------------------------
        df["Merchant_Freq"] = df.groupby("MerchantID")["TransactionID"].transform("count")

        # proxy: average spending at this merchant
        df["Merchant_Avg_Spend"] = df.groupby("MerchantID")["TransactionAmount"].transform("mean")

        # -----------------------------
        # Channel (ATM / Online) features
        # -----------------------------
        df["Channel_Online"] = (df["Channel"] == "Online").astype(int)
        df["Channel_ATM"] = (df["Channel"] == "ATM").astype(int)

        # -----------------------------
        # TransactionType (Debit/Credit)
        # -----------------------------
        df["Is_Credit"] = (df["TransactionType"] == "Credit").astype(int)
        df["Is_Debit"] = (df["TransactionType"] == "Debit").astype(int)

        # -----------------------------
        # "Paid Events"/"Large Checks" proxies
        # -----------------------------
        # large purchases: above mean + 2 std
        threshold = df["TransactionAmount"].mean() + 2 * df["TransactionAmount"].std()
        df["Is_Large_Purchase"] = (df["TransactionAmount"] >= threshold).astype(int)

        # “Paid event”: large + online
        df["Is_Paid_Event"] = (
            (df["TransactionAmount"] >= threshold) &
            (df["Channel"] == "Online")
        ).astype(int)

        # -----------------------------
        # Club features (you don't have clubs - so infer proxy clusters)
        # -----------------------------
        # "Size of club" → number of customers with same occupation
        df["Occupation_Group_Size"] = df.groupby("CustomerOccupation")["AccountID"].transform("count")

        # “Type of Club” → encode occupation
        df["Occupation_ID"] = df["CustomerOccupation"].astype("category").cat.codes

        # -----------------------------
        # Return numeric only
        # -----------------------------
        return df.select_dtypes(include=[np.number])

    # ----------------------------------------------------------
    # STEP 2: Fit Model
    # ----------------------------------------------------------
    def fit(self, df_raw: pd.DataFrame):
        """
        Train the Isolation Forest using raw transaction data.
        """

        X = self._preprocess_data(df_raw)
        X_scaled = self.scaler.fit_transform(X)

        self.model.fit(X_scaled)
        self.fitted = True
        return self

    # ----------------------------------------------------------
    # STEP 3: Predict Anomalies
    # ----------------------------------------------------------
    def predict(self, df_raw: pd.DataFrame) -> pd.DataFrame:
        """
        Returns anomaly scores + labels.
        -1 = anomaly, 1 = normal
        """

        if not self.fitted:
            raise ValueError("Model not fitted. Call fit() first.")

        X = self._preprocess_data(df_raw)
        X_scaled = self.scaler.transform(X)

        preds = self.model.predict(X_scaled)
        scores = self.model.decision_function(X_scaled)

        result = df_raw.copy()
        result["anomaly_label"] = preds
        result["anomaly_score"] = scores  # lower = more anomalous
        return result

In [ ]:
df = pd.read_csv("bank_transactions_data_2.csv")

In [ ]:
df.head()

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39


In [ ]:
iso = TransactionIsoForestModel()
iso.fit(df)

results = iso.predict(df)
print(results.head())

  TransactionID AccountID  TransactionAmount      TransactionDate  \
0      TX000001   AC00128              14.09  2023-04-11 16:29:14   
1      TX000002   AC00455             376.24  2023-06-27 16:44:19   
2      TX000003   AC00019             126.29  2023-07-10 18:16:08   
3      TX000004   AC00070             184.50  2023-05-05 16:32:11   
4      TX000005   AC00411              13.45  2023-10-16 17:51:24   

  TransactionType   Location DeviceID      IP Address MerchantID Channel  \
0           Debit  San Diego  D000380  162.198.218.92       M015     ATM   
1           Debit    Houston  D000051     13.149.61.4       M052     ATM   
2           Debit       Mesa  D000235  215.97.143.157       M009  Online   
3           Debit    Raleigh  D000187  200.13.225.150       M002  Online   
4          Credit    Atlanta  D000308    65.164.3.100       M091  Online   

   CustomerAge CustomerOccupation  TransactionDuration  LoginAttempts  \
0           70             Doctor                   81 

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

results["predicted_fraud"] = results["anomaly_label"].map({-1:1, 1:0})

In [ ]:
total_frauds = results["predicted_fraud"].sum()

In [ ]:
print(total_frauds)

510
